# **Baselines for Coding Benchmarks**

# Install Packages

In [ ]:
!pip install lm_eval
!pip install evalplus
!pip install datasets

# Download Models

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_TOKEN = os.environ["HF_TOKEN"]

BASE = "google/gemma-2-2b"
MODELS_IDS = {
    "base":      BASE,
    "math_ft":   "MergeBench/gemma-2-2b_math",
    "coding_ft": "MergeBench/gemma-2-2b_coding",
}
MAX_NEW = 512
SAMPLE_MERGED = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=HF_TOKEN,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    return tok, model

tok_base,  base_model  = load_model(MODELS_IDS["base"])
tok_math,  math_model  = load_model(MODELS_IDS["math_ft"])
tok_code,  code_model  = load_model(MODELS_IDS["coding_ft"])

assert tok_base.vocab_size == tok_math.vocab_size == tok_code.vocab_size, \
#    "Tokenizer vocab sizes must match!"

tok = tok_math
models = {
    "base":      base_model,
    "math_ft":   math_model,
    "coding_ft": code_model,
}
print("Models loaded:", list(models.keys()))

# Evaluate: Choose the model to be evaluated and run

In [ ]:
import json
import torch
from evalplus.data import get_human_eval_plus, get_mbpp_plus
from tqdm import tqdm

LIMIT = None

def generate(prompt, tokenizer, model, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_tokens = outputs[0][prompt_len:]
    completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    return completion


model = math_model
tok = tok_math


for benchmark, getter in [
    #("humaneval", get_human_eval_plus),
    ("mbpp", get_mbpp_plus),
]:
    print(f"\nLoading {benchmark} dataset...", end=" ", flush=True)
    problems = getter()
    print(f"done ({len(problems)} problems)")

    items = list(problems.items())
    if LIMIT is not None:
        items = items[:LIMIT]
        print(f"Limiting to {LIMIT}/{len(problems)} problems")

    samples = []
    failed = 0

    pbar = tqdm(
        items,
        desc=f"Generating [{benchmark}]",
        unit="problem",
        dynamic_ncols=True,
    )

    for task_id, problem in pbar:
        try:
            completion = generate(problem["prompt"], tok, model)
            samples.append({"task_id": task_id, "completion": completion})
        except Exception as e:
            failed += 1
            tqdm.write(f"{task_id} failed: {e}")
            samples.append({"task_id": task_id, "completion": ""})

        pbar.set_postfix(done=len(samples), failed=failed)

    out_file = f"samples_{benchmark}.jsonl"
    with open(out_file, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")

    print(f"\n[{benchmark}] {len(samples)} samples → {out_file}")
    if failed:
        print(f"{failed} problems failed and were saved as empty strings")

    print(f"Run: evalplus.evaluate --dataset {benchmark} --samples {out_file}\n")

In [ ]:
!python -m evalplus.evaluate --dataset mbpp --samples samples_mbpp.jsonl
!python -m evalplus.evaluate --dataset humaneval --samples samples_humaneval.jsonl

# **Baselines for Mathematics Benchmarks**

In [ ]:
import csv
import gc
import os
import re
import time
from collections import deque
from pathlib import Path
import multiprocessing as mp

import torch
from datasets import concatenate_datasets, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


HF_TOKEN = os.environ["HF_TOKEN"]
SEED = 0
MAX_NEW_TOKENS = 512
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

BASE = "google/gemma-2-2b"
MODEL_IDS = {
    "base": BASE,
    "math_ft": "MergeBench/gemma-2-2b_math",
    "coding_ft": "MergeBench/gemma-2-2b_coding",
}

MATH_DATASET = "DigitalLearningGmbH/MATH-lighteval"
GSM8K_DATASET = ("gsm8k", "main", "test")


if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")


def get_gpu_assignments(model_names):
    n_gpus = torch.cuda.device_count()
    if n_gpus == 0:
        return {name: None for name in model_names}
    if n_gpus == 1:
        return {name: 0 for name in model_names}

    ordered = list(model_names)
    return {
        ordered[0]: 0,
        ordered[1]: 1,
        ordered[2]: 0,
    }


def append_row(csv_path: Path, row: dict, fieldnames: list[str]):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not csv_path.exists() or csv_path.stat().st_size == 0
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)
        f.flush()
        os.fsync(f.fileno())


def overwrite_single_row_csv(csv_path: Path, row: dict, fieldnames: list[str]):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow(row)
        f.flush()
        os.fsync(f.fileno())


def load_existing_rows(csv_path: Path):
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return []
    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def extract_boxed(text: str) -> str:
    match = re.search(r"\\boxed\{([^}]*)\}", text)
    return match.group(1).strip() if match else text.strip()


def normalize_math_answer(ans: str) -> str:
    ans = ans.replace(" ", "").strip().lower()
    try:
        return str(float(ans))
    except ValueError:
        return ans


def score_math(pred: str, ref: str) -> bool:
    return normalize_math_answer(extract_boxed(pred)) == normalize_math_answer(extract_boxed(ref))


def extract_gsm8k_gold(answer: str) -> str:
    if "####" in answer:
        answer = answer.split("####")[-1]
    return normalize_gsm8k_answer(answer)


def extract_gsm8k_pred(text: str) -> str:
    if "####" in text:
        return normalize_gsm8k_answer(text.split("####")[-1])

    patterns = [
        r"(?:the answer is|final answer is|answer:)\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        r"([-+]?\d[\d,]*(?:\.\d+)?)\s*$",
    ]
    lowered = text.lower()
    for pattern in patterns:
        m = re.search(pattern, lowered)
        if m:
            return normalize_gsm8k_answer(m.group(1))
    return normalize_gsm8k_answer(text)


def normalize_gsm8k_answer(ans: str) -> str:
    ans = ans.strip().lower().replace(",", "")
    ans = ans.replace("$", "")
    ans = re.sub(r"\s+", "", ans)
    m = re.search(r"[-+]?\d+(?:\.\d+)?", ans)
    if m:
        num = m.group(0)
        try:
            return str(float(num)) if "." in num else str(int(num))
        except ValueError:
            return num
    return ans


def score_gsm8k(pred: str, ref: str) -> bool:
    return extract_gsm8k_pred(pred) == extract_gsm8k_gold(ref)


def build_math_prompt(problem: str) -> str:
    few_shot = """Solve the following math problem step by step. Put your final answer in \\boxed{}.
Problem: What is $2^{10}$?
Solution: $2^{10} = 1024$. The answer is $\\boxed{1024}$.
Problem: Simplify $\\frac{x^2 - 1}{x - 1}$.
Solution: $\\frac{x^2-1}{x-1} = \\frac{(x+1)(x-1)}{x-1} = x+1$. The answer is $\\boxed{x+1}$.
"""
    return few_shot + f"Problem: {problem}\nSolution:"


def build_gsm8k_prompt(question: str) -> str:
    return (
        "Solve the following grade-school math word problem step by step. "
        "End your response with '#### <final numeric answer>'.\n"
        f"Question: {question}\nAnswer:"
    )


def load_math_test(seed: int):
    try:
        ds = load_dataset(MATH_DATASET, "all", split="test")
    except Exception:
        subjects = [
            "algebra",
            "counting_and_probability",
            "geometry",
            "intermediate_algebra",
            "number_theory",
            "prealgebra",
            "precalculus",
        ]
        ds = concatenate_datasets([
            load_dataset(MATH_DATASET, subj, split="test") for subj in subjects
        ])
    return ds.shuffle(seed=seed)


def load_gsm8k_test(seed: int):
    name, subset, split = GSM8K_DATASET
    return load_dataset(name, subset, split=split).shuffle(seed=seed)


@torch.inference_mode()
def generate_text(model, tokenizer, prompt: str, device: torch.device) -> str:
    enc = tokenizer(prompt, return_tensors="pt", truncation=True)
    enc = {k: v.to(device) for k, v in enc.items()}

    output = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    gen_ids = output[0, enc["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


def evaluate_gsm8k_model(model_name: str, model, tokenizer, device: torch.device):
    answers_path = RESULTS_DIR / f"gsm8k_answers_{model_name}.csv"
    summary_path = RESULTS_DIR / f"gsm8k_summary_{model_name}.csv"
    fieldnames = [
        "idx", "model", "question", "prediction", "gold", "pred_answer", "gold_answer", "correct"
    ]

    existing = load_existing_rows(answers_path)
    done = {int(row["idx"]) for row in existing}
    correct = sum(int(row["correct"]) for row in existing)
    total = len(existing)

    ds = load_gsm8k_test(SEED)

    for idx, ex in enumerate(ds):
        if idx in done:
            continue

        prompt = build_gsm8k_prompt(ex["question"])
        pred = generate_text(model, tokenizer, prompt, device)
        pred_answer = extract_gsm8k_pred(pred)
        gold_answer = extract_gsm8k_gold(ex["answer"])
        ok = int(score_gsm8k(pred, ex["answer"]))

        row = {
            "idx": idx,
            "model": model_name,
            "question": ex["question"],
            "prediction": pred,
            "gold": ex["answer"],
            "pred_answer": pred_answer,
            "gold_answer": gold_answer,
            "correct": ok,
        }

        append_row(answers_path, row, fieldnames)

        correct += ok
        total += 1
        overwrite_single_row_csv(
            summary_path,
            {
                "model": model_name,
                "task": "gsm8k",
                "accuracy": correct / total,
                "correct": correct,
                "total": total,
            },
            ["model", "task", "accuracy", "correct", "total"],
        )


def evaluate_math_model(model_name: str, model, tokenizer, device: torch.device):
    answers_path = RESULTS_DIR / f"math_answers_{model_name}.csv"
    summary_path = RESULTS_DIR / f"math_summary_{model_name}.csv"
    fieldnames = [
        "idx", "model", "level", "type", "problem", "prediction", "gold", "correct"
    ]

    existing = load_existing_rows(answers_path)
    done = {int(row["idx"]) for row in existing}
    correct = sum(int(row["correct"]) for row in existing)
    total = len(existing)

    ds = load_math_test(SEED)

    for idx, ex in enumerate(ds):
        if idx in done:
            continue

        prompt = build_math_prompt(ex["problem"])
        pred = generate_text(model, tokenizer, prompt, device)
        ok = int(score_math(pred, ex["solution"]))

        row = {
            "idx": idx,
            "model": model_name,
            "level": ex.get("level", ""),
            "type": ex.get("type", ""),
            "problem": ex["problem"],
            "prediction": pred,
            "gold": ex["solution"],
            "correct": ok,
        }

        append_row(answers_path, row, fieldnames)

        correct += ok
        total += 1
        overwrite_single_row_csv(
            summary_path,
            {
                "model": model_name,
                "task": "math",
                "accuracy": correct / total,
                "correct": correct,
                "total": total,
            },
            ["model", "task", "accuracy", "correct", "total"],
        )


def run_one_model(model_name: str, model_id: str, gpu_id: int):
    if gpu_id is not None and torch.cuda.is_available():
        torch.cuda.set_device(gpu_id)
        device = torch.device(f"cuda:{gpu_id}")
    else:
        device = torch.device("cpu")

    dtype = torch.bfloat16 if device.type == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=HF_TOKEN,
        torch_dtype=dtype,
    ).to(device)
    model.eval()

    print(f"[{model_name}] running on {device} with dtype={dtype}")

    evaluate_gsm8k_model(model_name, model, tokenizer, device)
    evaluate_math_model(model_name, model, tokenizer, device)

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

def worker_main(model_name: str, model_id: str, gpu_id: int):
    run_one_model(model_name, model_id, gpu_id)


def run_all_models():
    model_names = list(MODEL_IDS.keys())
    assignments = get_gpu_assignments(model_names)
    n_gpus = torch.cuda.device_count()

    print("GPU assignments:", assignments)

    if n_gpus <= 1:
        for model_name in model_names:
            worker_main(model_name, MODEL_IDS[model_name], assignments[model_name])
        return

    max_parallel = min(n_gpus, 2)
    queue = deque((name, MODEL_IDS[name], assignments[name]) for name in model_names)
    active = []
    ctx = mp.get_context("spawn")

    while queue or active:
        while queue and len(active) < max_parallel:
            model_name, model_id, gpu_id = queue.popleft()
            p = ctx.Process(target=worker_main, args=(model_name, model_id, gpu_id))
            p.start()
            active.append((model_name, p))

        time.sleep(2)
        still_active = []
        for model_name, proc in active:
            if proc.is_alive():
                still_active.append((model_name, proc))
            else:
                proc.join()
                if proc.exitcode != 0:
                    raise RuntimeError(f"Worker for {model_name} failed with exit code {proc.exitcode}")
        active = still_active


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)
    run_all_models()